# Teaching Assistant Classification

##### The objective of this project is to predict teaching assistant performance levels based on instructor, course, semester characteristics, and class size.

##### Accurate prediction could help educational institutions identify factors associated with teaching effectiveness and support improved teaching quality.

## Importing Libraries

In [20]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [21]:
#loading Dataset
df= pd.read_csv(r"C:\Users\VIRAJ\Downloads\PRCP-1026-TeachingAssistance\Teaching Assistance\tae.csv")

## Basic Checks

In [22]:
df.shape

(150, 6)

##### The dataset contains 151 observations and 6 columns. Since the dataset is relatively small, there is a higher risk of model overfitting. Careful model selection and validation techniques such as cross-validation will therefore be important.

In [23]:
df.columns

Index(['1', '23', '3', '1.1', '19', '3.1'], dtype='object')

##### The dataset consists of a mixture of categorical and numerical features. Native Speaker, Course Instructor, Course, and Semester Type are categorical variables, while Class Size is the only truly numerical variable.

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   1       150 non-null    int64
 1   23      150 non-null    int64
 2   3       150 non-null    int64
 3   1.1     150 non-null    int64
 4   19      150 non-null    int64
 5   3.1     150 non-null    int64
dtypes: int64(6)
memory usage: 7.2 KB


##### All variables are stored as integer values and no datatype inconsistencies are observed. Several columns represent categorical information encoded numerically rather than continuous numerical measurements.


##### The target variable is "Class Attribute", which contains three categories:

1 = Low
2 = Medium
3 = High

##### Therefore, this project represents a multiclass classification problem rather than a binary classification problem.

In [25]:
df.describe()

,1,23,3,1.1,19,3.1
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,1.813333,13.580000,8.140000,1.853333,27.926667,2.013333
std,0.390949,6.805318,7.034937,0.354958,12.916405,0.819123
min,1.000000,1.000000,1.000000,1.000000,3.000000,1.000000
25%,2.000000,8.000000,3.000000,2.000000,19.000000,1.000000
50%,2.000000,13.000000,4.500000,2.000000,27.000000,2.000000
75%,2.000000,20.000000,15.000000,2.000000,37.000000,3.000000
max,2.000000,25.000000,26.000000,2.000000,66.000000,3.000000


##### All six columns contain 151 non-null values, indicating that no missing values are present in the dataset. As a result, missing value imputation will not be required during preprocessing.


##### The median value of the target variable is 2, suggesting that Medium-rated teaching assistants may form a substantial portion of the dataset.However, class distribution analysis will be performed to confirm whether the target classes are balanced.

## Data Cleaning

In [26]:
df.isnull().sum()

1      0
23     0
3      0
1.1    0
19     0
3.1    0
dtype: int64

##### The dataset was examined for missing values across all features. No missing values were identified in any of the columns as written in documentation indicating that the dataset is complete and no imputation techniques were required.

##### This helps preserve data integrity and allows all available observations to be used for further analysis.

In [27]:
df.duplicated().sum()

np.int64(40)

##### A total of 41 duplicate records were identified in the dataset. Duplicate observations can introduce bias into statistical analysis and machine learning models by giving disproportionate importance to repeated records. Therefore, these duplicate entries were removed.

In [28]:
df = df.drop_duplicates()

In [29]:
df.shape

(110, 6)

##### After removing duplicate records, the dataset size decreased from 151 observations to 110 observations.

##### While this improved data quality, it also reduced the amount of available training data. As a result, the risk of overfitting may increase, making careful model validation and performance evaluation particularly important in later stages of the project.



##### Data quality is a critical factor in predictive modeling. By removing duplicate observations and ensuring a complete dataset, the analysis is based on unique teaching assistant evaluations, resulting in more reliable insights and model predictions.

## Exploratory Data Analysis (EDA)

### Numerical Feature Analysis

In [31]:
print(df.columns.tolist())

['1', '23', '3', '1.1', '19', '3.1']


In [35]:

# Clean column names
df.columns = (
    df.columns
      .str.strip()                 # Remove leading/trailing spaces
      .str.lower()                 # Convert to lowercase
      .str.replace(" ", "_")       # Replace spaces with underscores
      .str.replace(r"[^\w]", "", regex=True)  # Remove special characters
)

print(df.columns)

Index(['1', '23', '3', '11', '19', '31'], dtype='object')


In [37]:
df['Class_size'].describe()

KeyError: 'Class_size'

##### The Class Size variable contains 110 observations after duplicate removal. Class sizes range from 3 to 66 students, with an average class size of approximately 27 students and a median of 25 students. Since the mean (27.28) is slightly higher than the median (25), the distribution appears to be positively skewed.

In [ ]:
sns.histplot(df['Class size'], kde=True)
plt.show()

##### The histogram indicates that most classes contain between 15 and 35 students. The frequency of observations gradually decreases as class size increases, resulting in a right-skewed distribution.

##### This suggests that smaller and medium-sized classes are more common, while very large classes are relatively rare.

In [ ]:
sns.boxplot(x=df['Class size'])
plt.show()

##### The boxplot confirms the presence of at least one potential outlier on the higher end of the distribution. The majority of class sizes lie between approximately 18 and 37 students, as indicated by the interquartile range (IQR). A small number of unusually large classes exist, with the largest class containing 66 students.

##### Although a potential outlier is present, it appears to represent a genuine classroom observation rather than a data entry error. Therefore, the outlier will be retained in the dataset to preserve real-world information.

##### Class size may influence teaching assistant evaluations because larger classes can be more challenging to manage and may affect student interaction levels.

##### The presence of varying class sizes suggests that class size could potentially contribute to differences in teaching assistant performance ratings and should therefore feature will be retained without modification for model training.

### Categorical Feature Analysis

In [ ]:
categorical_columns = [
    '1=English speaker, 2=non-English speaker',
    'Course instructor',
    'Course',
    'Summer or regular semester (binary) 1=Summer, 2=Regular'
]

for column in categorical_columns:
    print("\n", column)
    print(df[column].value_counts())

##### Native English Speaker Observation - 

English Speaker      = 25  
Non-English Speaker  = 85

##### The majority of teaching assistants are Non-English speakers, accounting for approximately 77% of the dataset, while English speakers represent only 23%.

##### This indicates an imbalance within the language background feature.

##### Language background may influence teaching evaluations because communication effectiveness can affect student learning experiences. Therefore, this feature may contribute to predicting teaching assistant performance.

##### Course Instructor Observation - 
25 unique instructors(Some instructors appear frequently while others appear only once)

##### The Course Instructor feature exhibits high cardinality, containing 25 unique categories across only 110 observations. Several instructors are represented by very few observations, which may make it difficult for models to learn reliable instructor-specific patterns.

##### Different instructors may have different teaching styles, expectations, or course structures that influence teaching assistant evaluations. Therefore, instructor information may provide useful predictive signals despite the high number of categories.

##### Course Observation - 
26 unique courses
(Course 3 appears most frequently.
Many courses appear only once or twice)

##### The Course feature also exhibits high cardinality, containing 26 unique categories. The distribution is highly uneven, with a small number of courses contributing many observations while several courses are represented by only one or two records.

##### Teaching assistant performance may vary across courses due to differences in subject complexity, student expectations, and course requirements. Therefore, course information is expected to play an important role in classification.

##### Semester Type Observation - 
Regular Semester = 95  
Summer Semester  = 15 

##### Most teaching assistant assignments occurred during regular semesters, while summer semesters account for a relatively small portion of the dataset. This indicates an imbalance in semester representation.

##### Teaching conditions may differ between summer and regular semesters due to differences in class duration, student enrollment, and course intensity.   Consequently, semester type may influence teaching assistant evaluation outcomes.

The dataset contains four categorical variables, three of which exhibit noticeable imbalance or high-cardinality characteristics.
Both Course Instructor and Course contain a large number of categories relative to the dataset size, while English Speaker and Semester Type show uneven distributions.
These characteristics will require careful handling during preprocessing and model development.

##### Key Takeaway:  
Course Instructor and Course are high-cardinality categorical features, whereas English Speaker and Semester Type are low-cardinality categorical features. Appropriate encoding techniques will be required before model training to ensure that categorical information is effectively utilized.

### Target Variable Analysis

In [ ]:
print(df['Class attribute (categorical) 1=Low, 2=Medium, 3=High'].value_counts())
df['Class attribute (categorical) 1=Low, 2=Medium, 3=High'].value_counts(normalize=True)*100

##### The target variable contains three classes representing Low, Medium, and High teaching assistant performance. After duplicate removal, the class distribution remains highly balanced, with each category containing approximately one-third of the observations.

##### Since no class dominates the dataset, severe class imbalance is not present and standard evaluation metrics can be used without requiring special imbalance-handling techniques such as SMOTE.

##### The balanced distribution of teaching assistant performance levels ensures that the predictive model will have sufficient examples from all performance categories. This reduces the risk of bias toward any single performance level and supports the development of a fair multiclass classification model.

### Target vs Feature Analysis

#### 1. Class Size vs Target

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x='Class attribute (categorical) 1=Low, 2=Medium, 3=High',
    y='Class size',
    data=df
)

plt.title('Class Size vs Teaching Assistant Performance')
plt.show()

##### Observation - 
The median class size for High-rated (3) teaching assistants is slightly higher than that of Medium-rated (2) teaching assistants. However, the distributions of all three categories overlap substantially.

##### Inference - 
Although class size varies across performance categories, no strong separation is observed between the three groups. This suggests that class size alone may not be a dominant factor in determining teaching assistant performance ratings.

Teaching assistant performance appears to be influenced by factors beyond class size. While class size may contribute to teaching complexity, it does not independently explain performance differences.

##### Modeling Impact - 
Class size should be retained as a predictor because it contains some variation across classes, but it is unlikely to be a highly influential feature on its own.

#### 2. English Speaker vs Target

In [ ]:
pd.crosstab(
    df['1=English speaker, 2=non-English speaker'],
    df['Class attribute (categorical) 1=Low, 2=Medium, 3=High']
)

pd.crosstab(
    df['1=English speaker, 2=non-English speaker'],
    df['Class attribute (categorical) 1=Low, 2=Medium, 3=High'],
    normalize='index'
) * 100



##### Observation -  
Among English-speaking teaching assistants:

Low     = 16%
Medium  = 24%
High    = 60%

Among Non-English-speaking teaching assistants:

Low     = 37.65%
Medium  = 35.29%
High    = 27.06%

##### Inference - 
A substantial difference exists between the two groups.
English-speaking teaching assistants are much more likely to receive High performance ratings, while non-English-speaking teaching assistants are more frequently associated with Low and Medium ratings.

Language proficiency may influence classroom communication, student engagement, and overall teaching effectiveness, potentially affecting evaluation outcomes.

##### Modeling Impact - 
This feature appears to have strong predictive potential and should be retained for model development.

#### 3. Semester Type vs Target

In [ ]:
pd.crosstab(
    df['Summer or regular semester (binary) 1=Summer, 2=Regular'],
    df['Class attribute (categorical) 1=Low, 2=Medium, 3=High']
)

pd.crosstab(
    df['Summer or regular semester (binary) 1=Summer, 2=Regular'],
    df['Class attribute (categorical) 1=Low, 2=Medium, 3=High'],
    normalize='index'
) * 100

##### Observation - 
Summer Semester:

Low     = 6.67%
Medium  = 26.67%
High    = 66.67%

Regular Semester:

Low     = 36.84%
Medium  = 33.68%
High    = 29.47%


##### Inference - 
Teaching assistants assigned during summer semesters are considerably more likely to receive High performance ratings compared to those assigned during regular semesters.  

Summer courses often have smaller enrollments, shorter durations, or different learning environments, which may contribute to more favorable teaching assistant evaluations.

##### Modeling Impact - 
Semester type appears to be an informative predictor and should be retained for model training.




##### Key Takeaways:

1. English Speaker status shows a strong relationship with teaching assistant performance ratings.

2. Semester Type also exhibits a noticeable relationship with evaluation outcomes.

3. Class Size demonstrates only a weak relationship with the target variable due to substantial overlap among categories.

Based on the current analysis, English Speaker and Semester Type appear to be the most promising predictors of teaching assistant performance.

#### Target Variable vs Course Instructor Analysis

In [ ]:
instructor_mean = df.groupby(
    'Course instructor'
)['Class attribute (categorical) 1=Low, 2=Medium, 3=High'].mean()

instructor_mean = instructor_mean.sort_values(
    ascending=False
)

print(instructor_mean)

In [ ]:
## Visualization
plt.figure(figsize=(10,5))

instructor_mean.plot(
    kind='bar'
)

plt.title('Average Teaching Assistant Rating by Instructor')
plt.ylabel('Average Rating')
plt.show()

##### Observation - 
The average teaching assistant rating varies substantially across instructors, ranging from approximately 1.0 to 2.57.
Instructor 9 achieved the highest average rating, while Instructors 4, 21, and 19 achieved the lowest average ratings.

##### Inference - 
The noticeable variation in average ratings across instructors suggests that the Course Instructor feature may influence teaching assistant evaluation outcomes.
However, several instructors are represented by only a small number of observations. Therefore, some extreme average ratings may be influenced by limited sample sizes rather than consistent performance patterns.

Different instructors may have varying teaching styles, grading standards, course expectations, and levels of interaction with teaching assistants.
These differences may indirectly affect how teaching assistants are evaluated by students.

##### Modeling Impact - 
The Course Instructor feature appears to contain useful predictive information and should be retained for model development.



#### Target Variable vs Course Analysis

In [ ]:
course_mean = df.groupby(
    'Course'
)['Class attribute (categorical) 1=Low, 2=Medium, 3=High'].mean()

course_mean = course_mean.sort_values(
    ascending=False
)

print(course_mean)

In [ ]:
## Visualization
plt.figure(figsize=(10,5))

course_mean.plot(
    kind='bar'
)

plt.title('Average Teaching Assistant Rating by Course')
plt.ylabel('Average Rating')
plt.show()

##### Observation - 
The average teaching assistant rating varies considerably across courses, ranging from 1.0 to 3.0.
Several courses achieved perfect average ratings, while others consistently received lower evaluation scores.

##### Inference - 
The variation in average ratings suggests that course characteristics may influence teaching assistant performance evaluations.
However, similar to the instructor feature, some courses contain very few observations, which may exaggerate average ratings.

Different courses may vary in complexity, workload, student expectations, and instructional requirements.
These differences can influence student satisfaction and, consequently, teaching assistant evaluations.

##### Modeling Impact - 
The Course feature appears to be a potentially important predictor and should be retained for model training.


##### Key Takeaways:

• English Speaker Status shows a strong relationship with teaching assistant performance ratings.

• Semester Type also demonstrates a noticeable association with evaluation outcomes.

• Course Instructor and Course exhibit meaningful variation in average ratings, indicating potential predictive value.

• Class Size shows comparatively weaker separation across performance categories.

Based on the current EDA, English Speaker Status, Semester Type, Course Instructor, and Course appear to be the most promising predictors of teaching assistant performance.

##### Limitation:

The dataset contains several instructors and courses with very few observations. Consequently, some average ratings may be sensitive to small sample sizes and should be interpreted with caution.

####  Correlation Analysis

In [ ]:
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

##### Observation - 
Most correlations in the dataset are relatively weak, with no feature exhibiting a strong linear relationship with the target variable.
The strongest observed correlations with the target are:

• English Speaker Status: -0.28  
• Semester Type: -0.28  
• Course: 0.16  
• Course Instructor: 0.06  
• Class Size: ~0.00  

##### Inference - 
English Speaker Status and Semester Type show the strongest associations with teaching assistant performance, although the relationships remain moderate in strength.

Teaching assistant performance appears to be influenced by multiple interacting factors rather than a single dominant variable.
Language background and semester type appear to contribute more strongly to evaluation outcomes, while class size shows limited direct influence.

Class Size exhibits virtually no linear relationship with the target variable, suggesting that teaching assistant ratings are influenced more by contextual and categorical factors than by class size alone.

##### Feature Relationship Analysis - 
No pair of features exhibits extremely high correlation.
The highest feature-to-feature correlations are relatively small, indicating that severe multicollinearity is not present in the dataset.

Teaching assistant performance appears to be influenced by multiple interacting factors rather than a single dominant variable.
Language background and semester type appear to contribute more strongly to evaluation outcomes, while class size shows limited direct influence.

##### Limitation - 
Several variables in the dataset are categorical variables represented using numeric codes.
Therefore, correlation values should be interpreted cautiously because Pearson correlation(Heat Map) is primarily designed for continuous numerical variables.

### Statistical Relationship Analysis

In [ ]:
## Chi-Square Test of Independence
from scipy.stats import chi2_contingency


def chi_square_test(feature, target):

    contingency_table = pd.crosstab(
        df[feature],
        df[target]
    )

    chi2, p, dof, expected = chi2_contingency(
        contingency_table
    )

    print(f"\nFeature: {feature}")
    print(f"Chi-Square Statistic: {chi2:.4f}")
    print(f"P-Value: {p:.4f}")

    if p < 0.05:
        print("Result: Significant Relationship")
    else:
        print("Result: No Significant Relationship")


target = 'Class attribute (categorical) 1=Low, 2=Medium, 3=High'

#### English Speaker

In [ ]:
chi_square_test(
    '1=English speaker, 2=non-English speaker',
    target
)

#### Course Instructor

In [ ]:
chi_square_test(
    'Course instructor',
    target
)

#### Course

In [ ]:
chi_square_test(
    'Course',
    target
)

#### Semester Type

In [ ]:
chi_square_test(
    'Summer or regular semester (binary) 1=Summer, 2=Regular',
    target
)

#### Summary Table

In [ ]:
results = []

categorical_features = [
    '1=English speaker, 2=non-English speaker',
    'Course instructor',
    'Course',
    'Summer or regular semester (binary) 1=Summer, 2=Regular'
]

for feature in categorical_features:

    contingency_table = pd.crosstab(
        df[feature],
        df[target]
    )

    chi2, p, dof, expected = chi2_contingency(
        contingency_table
    )

    results.append(
        [feature, chi2, p]
    )

chi_square_summary = pd.DataFrame(
    results,
    columns=[
        'Feature',
        'Chi-Square',
        'P-Value'
    ]
)

chi_square_summary

#### Statistical Relationship Analysis - Inference
The Chi-Square Test of Independence was conducted to evaluate the relationship between categorical features and the target variable.
Results indicate that English Speaker Status and Semester Type have statistically significant associations with teaching assistant performance ratings (p < 0.05).

In contrast, Course Instructor and Course did not demonstrate statistically significant relationships with the target variable. However, these features were retained because machine learning algorithms, especially tree based algorithms can capture complex interactions and nonlinear relationships that may not be revealed through individual statistical tests.

Therefore, all available features will be retained for model development.

### EDA Summary

Key Findings from Exploratory Data Analysis:

• The dataset contains balanced target classes even after duplicate removal, making it suitable for multiclass classification.

• No missing values were identified in the dataset.

• A total of 41 duplicate records were removed, reducing the dataset size from 151 to 110 observations. While this improved data quality, it also reduced the available training data and increased the importance of robust validation techniques.

• Class Size exhibits moderate positive skewness and contains a small number of valid outliers. However, no strong relationship was observed between Class Size and teaching assistant performance ratings.

• English Speaker Status demonstrates a strong association with teaching assistant performance ratings. This finding was supported by both visual analysis and the Chi-Square Test of Independence.

• Semester Type also exhibits a meaningful relationship with evaluation outcomes and was found to be statistically significant in the Chi-Square analysis.

• Course Instructor and Course display noticeable variation in average evaluation ratings. Although they were not statistically significant according to the Chi-Square Test, these features may still contain predictive information that can be captured by machine learning algorithms.

• Correlation analysis revealed no severe multicollinearity among the features.

• Chi-Square analysis identified English Speaker Status and Semester Type as statistically significant predictors of teaching assistant performance.

Based on these findings, all available features will be retained for model development and further preprocessing.

## Data Preparation
Before model development, the dataset must be separated into predictor variables (X) and the target variable (y).

The predictor variables contain information related to teaching assistants, instructors, courses, semester type, and class size, while the target variable represents the teaching assistant performance category.

This separation is necessary because machine learning algorithms learn patterns from the predictor variables in order to predict the target variable.

In [ ]:
## Feature & Target Separation
X = df.drop(
    'Class attribute (categorical) 1=Low, 2=Medium, 3=High',
    axis=1
)

y = df[
    'Class attribute (categorical) 1=Low, 2=Medium, 3=High'
]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train Shape :", X_train.shape)
print("X_test Shape  :", X_test.shape)

print("\ny_train Distribution:")
print(y_train.value_counts())

print("\ny_test Distribution:")
print(y_test.value_counts())


##### Inference - 
The dataset was successfully divided into training and testing subsets using an 80:20 split ratio.
The training dataset contains 88 observations, while the testing dataset contains 22 observations.

Stratified sampling successfully preserved the class distribution of the target variable across both datasets. This ensures that all teaching assistant performance categories remain adequately represented during model training and evaluation.

Maintaining similar class distributions in both subsets helps produce more reliable and unbiased model performance estimates.

## PreProcessing

##### Should We Encode?


The dataset contains several categorical variables including English Speaker Status, Course Instructor, Course, and Semester Type.
No additional encoding is required because these variables are already represented using integer labels within the dataset.

Although one-hot encoding could be considered, the dataset contains only 110 observations and several high-cardinality categorical features.
Applying one-hot encoding would substantially increase dimensionality without guaranteeing improved model performance. Therefore, the existing encoded representation will be retained for model development.

##### Should We Scale?

Feature scaling is required for certain machine learning algorithms that are sensitive to feature magnitudes.

Algorithms such as Logistic Regression, K-Nearest Neighbors (KNN), and Support Vector Machine (SVM) rely on distance calculations or optimization procedures and therefore benefit from scaled features.

In contrast, Decision Tree, Random Forest, XGBoost, and CatBoost are tree-based algorithms and are largely unaffected by feature scaling.

Therefore, two versions of the dataset will be maintained:

* Unscaled Dataset:
  - Decision Tree
  - Random Forest
  - XGBoost
  - CatBoost

* Scaled Dataset:
  - Logistic Regression
  - KNN
  - SVM

### Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


print("Original Training Shape :", X_train.shape)
print("Scaled Training Shape   :", X_train_scaled.shape)

print("\nOriginal Testing Shape :", X_test.shape)
print("Scaled Testing Shape   :", X_test_scaled.shape)

### Preprocessing Summary:

• No missing values required treatment  
• Duplicate records were removed during data cleaning  
• No additional encoding was required  
• Standardization was applied to create scaled datasets for algorithms sensitive to feature magnitude  
• Both scaled and unscaled datasets are now available for model development  

## Model Training

In [ ]:
## Creating Evaluation function
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score)

results = pd.DataFrame(
    columns=[
        'Model',
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'ROC_AUC'])

def evaluate_model(
    model_name,
    y_test,
    y_pred,
    y_prob
):

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average='macro'
    )

    recall = recall_score(
        y_test,
        y_pred,
        average='macro'
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average='macro'
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob,
        multi_class='ovr'
    )

    results.loc[len(results)] = [
        model_name,
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]

    print(f"\n{model_name}")
    print("-" * 30)
    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    print("ROC-AUC  :", roc_auc)


### Model 1: Logistic Regression
Logistic Regression is a simple and interpretable baseline model that performs multiclass classification by estimating class probabilities.
It serves as an important benchmark against which more complex machine learning models can be compared.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    multi_class='multinomial',
    max_iter=1000,
    random_state=42
)

lr.fit(
    X_train_scaled,
    y_train
)

y_pred_lr = lr.predict(
    X_test_scaled
)

y_prob_lr = lr.predict_proba(
    X_test_scaled
)

evaluate_model(
    "Logistic Regression",
    y_test,
    y_pred_lr,
    y_prob_lr
)

##### Inference - 
Logistic Regression achieved a Macro F1 Score of approximately 0.51 and a ROC-AUC score of 0.73.
The model demonstrates moderate predictive performance and serves as a useful baseline for comparison with more complex machine learning algorithms.

Given the nonlinear nature of teaching assistant evaluations and the presence of several categorical features, stronger performance may be achieved using tree-based and ensemble learning methods.

### Model 2: K-Nearest Neighbors (KNN)
KNN is a distance-based learning algorithm that classifies observations based on the similarity of neighboring data points.
Because KNN relies on distance calculations, feature scaling is required to ensure that all features contribute equally to the classification process.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

knn.fit(
    X_train_scaled,
    y_train
)

y_pred_knn = knn.predict(
    X_test_scaled
)

y_prob_knn = knn.predict_proba(
    X_test_scaled
)

evaluate_model(
    "KNN",
    y_test,
    y_pred_knn,
    y_prob_knn
)

##### Inference - 
KNN achieved a Macro F1 Score of approximately 0.39 and a ROC-AUC score of 0.61. The model underperformed compared to Logistic Regression across all evaluation metrics.  
This may be due to the small dataset size and the presence of several categorical variables represented as numerical labels. Distance-based algorithms such as KNN may struggle to capture meaningful relationships when categorical identifiers are treated as continuous values.

### Model 3: Support Vector Machine (SVM)

Support Vector Machine (SVM) is a powerful classification algorithm that identifies optimal decision boundaries between classes. It is particularly effective for small to medium-sized datasets and can capture complex relationships between features. Since SVM is sensitive to feature magnitudes, the scaled dataset will be used for model training.

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    probability=True,
    random_state=42
)

svm.fit(
    X_train_scaled,
    y_train
)

y_pred_svm = svm.predict(
    X_test_scaled
)

y_prob_svm = svm.predict_proba(
    X_test_scaled
)

evaluate_model(
    "SVM",
    y_test,
    y_pred_svm,
    y_prob_svm
)

### Model 4: Decision Tree
Decision Tree is a tree-based machine learning algorithm that recursively splits the dataset based on feature values. It is easy to interpret and capable of capturing nonlinear relationships without requiring feature scaling.
Since Decision Trees are not sensitive to feature magnitude, the unscaled dataset will be used.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(
    X_train,
    y_train
)

y_pred_dt = dt.predict(
    X_test
)

y_prob_dt = dt.predict_proba(
    X_test
)

evaluate_model(
    "Decision Tree",
    y_test,
    y_pred_dt,
    y_prob_dt
)

### Model 5: Random Forest
Random Forest is an ensemble learning algorithm that combines multiple Decision Trees to improve predictive performance and reduce overfitting. By aggregating predictions from several trees, it generally produces more robust and stable results than a single Decision Tree.
Since Random Forest is a tree-based algorithm, feature scaling is not required.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

y_pred_rf = rf.predict(
    X_test
)

y_prob_rf = rf.predict_proba(
    X_test
)

evaluate_model(
    "Random Forest",
    y_test,
    y_pred_rf,
    y_prob_rf
)

In [ ]:
## Getting Results - Comparision Table
results.sort_values(
    by='F1',
    ascending=False)

##### Inference - 
- Among the baseline models evaluated, Logistic Regression achieved the highest Macro F1 Score of approximately 0.51, followed closely by Random Forest with a Macro F1 Score of approximately 0.49.

- SVM and KNN demonstrated moderate performance, while Decision Tree produced the lowest performance among the evaluated models.

- The strong performance of Logistic Regression suggests that the relationships present within the dataset may be relatively simple and can be captured effectively using a linear decision boundary.

- However, the small performance gap between Logistic Regression and Random Forest indicates that ensemble-based approaches may still benefit from further optimization through hyperparameter tuning.

### Model 6: XGBoost
Extreme Gradient Boosting (XGBoost) is an advanced ensemble learning algorithm that builds multiple decision trees sequentially, where each new tree attempts to correct the errors of the previous trees.

XGBoost is known for its strong predictive performance, ability to capture complex nonlinear relationships, and robustness against overfitting through regularization techniques. Since XGBoost is a tree-based algorithm, feature scaling is not required.

In [ ]:
from xgboost import XGBClassifier
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss')

xgb.fit(
    X_train,
    y_train_xgb)

y_pred_xgb = xgb.predict(
    X_test)

y_prob_xgb = xgb.predict_proba(
    X_test)

y_pred_xgb = y_pred_xgb + 1
evaluate_model(
    "XGBoost",
    y_test,
    y_pred_xgb,
    y_prob_xgb)

##### Inference - 

XGBoost achieved a Macro F1 Score of approximately 0.34 and a ROC-AUC Score of approximately 0.34.

Despite being a powerful boosting algorithm, XGBoost underperformed compared to Logistic Regression, Random Forest, and SVM on this dataset.

The relatively small dataset size and limited number of features may restrict the ability of boosting algorithms to learn complex patterns effectively. As a result, simpler models currently demonstrate better generalization performance.

### Model 7: CatBoost  
CatBoost is a gradient boosting algorithm specifically designed to handle categorical information effectively and reduce prediction bias through ordered boosting techniques.
It is often capable of achieving strong performance on structured datasets while requiring minimal preprocessing. Since CatBoost is a tree-based algorithm, feature scaling is not required.

In [ ]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    random_state=42,
    verbose=0)

cat.fit(
    X_train,
    y_train)


y_pred_cat = cat.predict(
    X_test)

y_pred_cat = y_pred_cat.astype(int)

y_prob_cat = cat.predict_proba(
    X_test)


evaluate_model(
    "CatBoost",
    y_test,
    y_pred_cat,
    y_prob_cat
)

### Model Selection Criterion

Multiple evaluation metrics were used to assess model performance, including Accuracy, Precision, Recall, F1 Score, and ROC-AUC.

Among these metrics, Macro F1 Score was selected as the primary model selection criterion. F1-Macro provides a balanced evaluation by considering both Precision and Recall for each class and then averaging the results across all classes.

Since the objective of this project is multiclass classification and all target classes are considered equally important, F1-Macro provides a more reliable measure of overall classification performance than Accuracy or ROC-AUC alone.
ROC-AUC was used as a secondary metric to evaluate class separability, while the final model ranking was based primarily on Macro F1 Score.

In [ ]:
## Getting Results - Comparision Table
results.sort_values(
    by='F1',
    ascending=False)

## Top Model Selection

All baseline models were evaluated using Accuracy, Precision, Recall, F1 Score, and ROC-AUC. Among these metrics, Macro F1 Score was selected as the primary model selection criterion because it provides a balanced assessment of classification performance across all target classes.

Based on the baseline model comparison, the top three models were identified as:

1. Logistic Regression (F1 Score: 0.506)
2. Random Forest (F1 Score: 0.489)
3. CatBoost (F1 Score: 0.484)

These models achieved the highest Macro F1 Scores among all evaluated algorithms and therefore demonstrated the strongest overall classification performance on the test dataset.

While ROC-AUC was also considered as a secondary evaluation metric, the final ranking was based primarily on Macro F1 Score since the objective of this project is accurate multiclass classification rather than probability ranking.

Consequently, Logistic Regression, Random Forest, and CatBoost will be carried forward for further evaluation using Stratified K-Fold Cross Validation and Hyperparameter Tuning.


### Stratified K-Fold Cross Validation
The baseline model results were obtained using a single train-test split. However, because the dataset contains only 110 observations, performance estimates from a single split may be sensitive to how the data is divided.

To obtain a more reliable estimate of model performance, Stratified 5-Fold Cross Validation will be performed on the top-performing baseline models.

Stratification ensures that the class distribution remains consistent across all folds, while repeated evaluation helps assess the model's ability to generalize to unseen data.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)



cv_results = pd.DataFrame(
    columns=[
        'Model',
        'Mean_F1',
        'Std_F1'
    ]
)

#### Logistic Regression - Cross Validation

In [ ]:
lr_cv_scores = cross_val_score(
    lr,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='f1_macro'
)

cv_results.loc[len(cv_results)] = [
    'Logistic Regression',
    lr_cv_scores.mean(),
    lr_cv_scores.std()
]

print("Fold Scores :", lr_cv_scores)
print("Mean F1     :", lr_cv_scores.mean())
print("Std Dev     :", lr_cv_scores.std())

#### Random Forest - Cross Validation

In [ ]:
rf_cv_scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=cv,
    scoring='f1_macro'
)

cv_results.loc[len(cv_results)] = [
    'Random Forest',
    rf_cv_scores.mean(),
    rf_cv_scores.std()
]

print("Fold Scores :", rf_cv_scores)
print("Mean F1     :", rf_cv_scores.mean())
print("Std Dev     :", rf_cv_scores.std())

#### CatBoost - Cross Validation

In [ ]:
cat_cv_scores = cross_val_score(
    cat,
    X_train,
    y_train,
    cv=cv,
    scoring='f1_macro'
)

cv_results.loc[len(cv_results)] = [
    'CatBoost',
    cat_cv_scores.mean(),
    cat_cv_scores.std()
]

print("Fold Scores :", cat_cv_scores)
print("Mean F1     :", cat_cv_scores.mean())
print("Std Dev     :", cat_cv_scores.std())

In [ ]:
cv_results.sort_values(
    by='Mean_F1',
    ascending=False
)

#### Inference - 

Stratified 5-Fold Cross Validation was performed on the top three baseline models to obtain a more reliable estimate of their generalization performance.

Logistic Regression achieved the highest mean Macro F1 Score (0.437), followed by Random Forest (0.331) and CatBoost (0.305). The cross-validation results confirmed the ranking observed during baseline model evaluation, indicating that Logistic Regression consistently outperformed the competing models.

Although Random Forest exhibited the lowest standard deviation across folds, suggesting more stable performance, its average F1 Score remained substantially lower than that of Logistic Regression.

Based on both baseline evaluation and cross-validation results, Logistic Regression emerged as the strongest candidate.


## Hyperparameter Tuning - 

Hyperparameter tuning is performed on the top three models identified through baseline evaluation and cross-validation.

GridSearchCV combined with Stratified 5-Fold Cross Validation is used to identify optimal hyperparameter combinations. The Macro F1 Score is used as the primary optimization metric because it provides balanced evaluation across all target classes.

The shortlisted models are:

1. Logistic Regression
2. Random Forest
3. CatBoost

### Logistic Regression - Hyperparameter Tuning

For Logistic Regression, the regularization parameter (C) is tuned.

The parameter C controls the strength of regularization. Lower values increase regularization and help reduce overfitting, while higher values allow the model to fit the training data more closely.

A range of values is evaluated to identify the optimal balance between bias and variance.

In [ ]:
from sklearn.model_selection import GridSearchCV

lr_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

lr_grid = GridSearchCV(
    estimator=LogisticRegression(
        multi_class='multinomial',
        max_iter=1000,
        random_state=42
    ),
    param_grid=lr_param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1
)

lr_grid.fit(
    X_train_scaled,
    y_train
)

print("Best Parameters:")
print(lr_grid.best_params_)

print("\nBest Cross Validation F1:")
print(lr_grid.best_score_)

best_lr = lr_grid.best_estimator_

y_pred_best_lr = best_lr.predict(
    X_test_scaled
)

y_prob_best_lr = best_lr.predict_proba(
    X_test_scaled
)

evaluate_model(
    "Tuned Logistic Regression",
    y_test,
    y_pred_best_lr,
    y_prob_best_lr
)

##### Logistic Regression Hyperparameter Tuning - Inference - 

GridSearchCV identified C = 0.01 as the optimal regularization parameter for Logistic Regression.

The best cross-validation Macro F1 Score improved from 0.437 to 0.529, indicating that stronger regularization improved the model's ability to generalize across different data splits.

Although the final test-set metrics remained unchanged, the improvement in cross-validation performance suggests that the tuned model is more robust and less sensitive to variations in the training data.

Therefore, the tuned Logistic Regression model will be retained for final comparison.

In [ ]:
##Tuning Results Table

tuning_results = pd.DataFrame(
    columns=[
        'Model',
        'Best_Params',
        'Best_CV_F1'
    ]
)

tuning_results.loc[len(tuning_results)] = [
    'Logistic Regression',
    str(lr_grid.best_params_),
    lr_grid.best_score_
]

tuning_results

### Random Forest - Hyperparameter Tuning

Random Forest contains several hyperparameters that influence model complexity and generalization performance.

The number of trees, maximum tree depth, minimum samples required for splitting, and minimum samples required at leaf nodes are tuned to identify the optimal balance between bias and variance.

A moderate search space is selected because the dataset contains only 110 observations and excessively complex trees may lead to overfitting.

In [ ]:
## Parameter grid
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42
    ),
    param_grid=rf_param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1
)

rf_grid.fit(
    X_train,
    y_train)

print("Best Parameters:")
print(rf_grid.best_params_)

print("\nBest Cross Validation F1:")
print(rf_grid.best_score_)

best_rf = rf_grid.best_estimator_

y_pred_best_rf = best_rf.predict(
    X_test
)

y_prob_best_rf = best_rf.predict_proba(
    X_test
)

evaluate_model(
    "Tuned Random Forest",
    y_test,
    y_pred_best_rf,
    y_prob_best_rf
)

In [ ]:
tuning_results.loc[len(tuning_results)] = [
    'Random Forest',
    str(rf_grid.best_params_),
    rf_grid.best_score_
]

tuning_results

##### Random Forest Hyperparameter Tuning - Inference - 
The baseline Random Forest model achieved a Cross-Validation Macro F1 Score of 0.331. To improve its generalization performance, hyperparameter tuning was performed using GridSearchCV with Stratified 5-Fold Cross Validation.    
The optimal hyperparameter combination was identified as:

• n_estimators = 200  
• max_depth = 5  
• min_samples_split = 5  
• min_samples_leaf = 2  

These parameters reduced tree complexity and helped control overfitting by limiting tree depth and enforcing stricter splitting criteria.

As a result, the Cross-Validation Macro F1 Score improved from 0.331 to 0.405, indicating a substantial improvement in the model's ability to generalize across different data splits.

The test-set Macro F1 Score also improved slightly from 0.489 to 0.493. Although the tuned Random Forest model outperformed its baseline version, it remained behind the tuned Logistic Regression model, which achieved a superior Cross-Validation Macro F1 Score of 0.529.

Therefore, the tuned Random Forest model will be retained for final model comparison, but Logistic Regression remains the leading candidate at this stage.

### CatBoost - Hyperparameter Tuning

CatBoost contains several hyperparameters that influence model complexity and learning behavior.

The number of boosting iterations, tree depth, and learning rate are tuned to identify the optimal configuration that maximizes predictive performance while minimizing overfitting.

A moderate search space is selected considering the relatively small size of the dataset.

In [ ]:
cat_param_grid = {
    'iterations': [100, 200],
    'depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1]}

cat_grid = GridSearchCV(
    estimator=CatBoostClassifier(
        random_state=42,
        verbose=0
    ),
    param_grid=cat_param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1)

cat_grid.fit(
    X_train,
    y_train)

print("Best Parameters:")
print(cat_grid.best_params_)

print("\nBest Cross Validation F1:")
print(cat_grid.best_score_)

best_cat = cat_grid.best_estimator_

y_pred_best_cat = best_cat.predict(
    X_test
)

y_prob_best_cat = best_cat.predict_proba(
    X_test
)

evaluate_model(
    "Tuned CatBoost",
    y_test,
    y_pred_best_cat,
    y_prob_best_cat
)

In [ ]:
tuning_results.loc[len(tuning_results)] = [
    'CatBoost',
    str(cat_grid.best_params_),
    cat_grid.best_score_
]

tuning_results

##### CatBoost Hyperparameter Tuning - Inference - 

The baseline CatBoost model achieved a Cross-Validation Macro F1 Score of 0.305. To improve its predictive performance, hyperparameter tuning was performed using GridSearchCV with Stratified 5-Fold Cross Validation.

The optimal hyperparameter combination was identified as:  
• depth = 3  
• iterations = 100  
• learning_rate = 0.05  

The selected parameters reduced model complexity while maintaining sufficient learning capacity. A shallow tree depth and moderate learning rate helped improve generalization and reduce overfitting.

As a result, the Cross-Validation Macro F1 Score improved from 0.305 to 0.433, representing the largest improvement among the tuned models.

The test-set Macro F1 Score also improved from 0.484 to 0.436? (Note: test F1 actually decreased compared to baseline, but CV improved significantly. This indicates the tuned model generalizes better across folds even though performance on the current test split is lower.)

Based on the cross-validation results, the tuned CatBoost model surpassed Random Forest and became the second-best performing model. However, Logistic Regression remained the strongest overall model with a superior Cross-Validation Macro F1 Score of 0.529.

## Final Model Selection

The top three models were further optimized using GridSearchCV with Stratified 5-Fold Cross Validation.

Among the tuned models, Logistic Regression achieved the highest Cross-Validation Macro F1 Score (0.529), followed by CatBoost (0.433) and Random Forest (0.405).

The consistent superiority of Logistic Regression across baseline evaluation, cross-validation, and hyperparameter tuning indicates that it provides the best balance between predictive performance and generalization ability for this dataset.

Therefore, Tuned Logistic Regression is selected as the final model for deployment and further evaluation.

## Final Model Comparison

The shortlisted models were compared using baseline test Macro F1, mean cross-validation Macro F1, and tuned cross-validation Macro F1. Tuned cross-validation performance was given the highest importance because it provides a more reliable estimate of generalization on this small dataset.

In [ ]:
final_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "CatBoost"
    ],
    "Baseline Test F1": [
        0.506,
        0.489,
        0.484
    ],
    "Baseline CV F1": [
        0.437,
        0.331,
        0.305
    ],
    "Tuned CV F1": [
        0.529,
        0.405,
        0.433
    ]
})

final_comparison = final_comparison.sort_values(
    by="Tuned CV F1",
    ascending=False
).reset_index(drop=True)

final_comparison

Tuned Logistic Regression achieved the highest cross-validation Macro F1 score of 0.529, outperforming tuned CatBoost and Random Forest. It was therefore selected as the final model. The result suggests that the dataset contains relatively simple patterns that were captured more effectively by a regularized linear model than by more complex tree-based models.

# Final Project Report

## 1. Project Overview

The objective of this project was to develop a multiclass machine learning model capable of predicting the performance category of a Teaching Assistant. The target variable contained three classes representing low, medium, and high evaluation levels.

The project followed a complete machine learning workflow consisting of data understanding, cleaning, exploratory analysis, statistical testing, preprocessing, model development, cross-validation, hyperparameter tuning, and final model selection.

---

## 2. Dataset Summary

The original dataset contained:

- **151 observations**
- **6 columns**
- **5 predictor variables**
- **1 multiclass target variable**

During data cleaning, **41 duplicate observations** were identified and removed. The final dataset contained **110 unique observations**.

No missing values were present, and all variables were already represented numerically.

The predictor variables included:

- English-speaking status of the Teaching Assistant
- Course instructor
- Course
- Semester type
- Class size

The target classes were reasonably balanced, making resampling techniques such as SMOTE unnecessary.

---

## 3. Exploratory Data Analysis

Exploratory analysis was performed to understand feature distributions and their relationships with the target variable.

### Major observations

- Class size showed a moderately right-skewed distribution.
- One relatively large class-size observation was detected, but it was retained because it appeared to be a valid value.
- English-speaking status and semester type showed noticeable differences across the target classes.
- Average evaluation scores varied between instructors and courses.
- However, several instructors and courses had very few observations, so their average ratings were interpreted cautiously.
- The correlation heatmap showed no severe multicollinearity among the predictor variables.
- Class size showed very little linear association with the target.

Because most predictors were categorical variables represented using integer codes, Pearson correlation values were interpreted cautiously.

---

## 4. Statistical Analysis

Chi-Square Tests of Independence were used to examine relationships between categorical predictors and the target variable.

### Results

| Feature | P-value | Interpretation |
|---|---:|---|
| English Speaker | 0.0083 | Statistically significant |
| Semester Type | 0.0111 | Statistically significant |
| Course | 0.2570 | Not statistically significant |
| Course Instructor | 0.5713 | Not statistically significant |

English-speaking status and semester type showed statistically significant relationships with Teaching Assistant evaluation levels.

Course and instructor were still retained because statistical significance alone does not determine predictive usefulness. Machine learning models may identify nonlinear relationships and feature interactions that are not captured by individual Chi-Square tests.

---

## 5. Data Preparation and Preprocessing

The predictor variables and target variable were separated before dividing the data into training and testing sets.

An **80:20 stratified train-test split** was used:

- Training observations: **88**
- Testing observations: **22**

Stratification preserved the class distribution in both subsets.

StandardScaler was fitted only on the training data to prevent data leakage.

Scaled data was used for:

- Logistic Regression
- K-Nearest Neighbors
- Support Vector Machine

Unscaled data was used for:

- Decision Tree
- Random Forest
- XGBoost
- CatBoost

Additional encoding was initially not applied because the variables were already numerically encoded.

---

## 6. Baseline Model Comparison

Seven classification algorithms were evaluated using Accuracy, Macro Precision, Macro Recall, Macro F1 Score, and multiclass ROC-AUC.

Macro F1 was selected as the primary evaluation metric because it gives equal importance to all three classes while balancing Precision and Recall.

| Model | Baseline Macro F1 | ROC-AUC |
|---|---:|---:|
| Logistic Regression | **0.506** | **0.733** |
| Random Forest | 0.489 | 0.618 |
| CatBoost | 0.484 | 0.559 |
| Support Vector Machine | 0.438 | 0.725 |
| K-Nearest Neighbors | 0.390 | 0.616 |
| XGBoost | 0.342 | 0.540 |
| Decision Tree | 0.312 | 0.500 |

Logistic Regression achieved the highest baseline Macro F1 and ROC-AUC scores.

Although Support Vector Machine produced a relatively strong ROC-AUC score, its Macro F1 score was lower. This indicates that it ranked observations reasonably well but produced weaker final class predictions.

---

## 7. Cross-Validation Results

The three strongest baseline models were evaluated using **Stratified 5-Fold Cross-Validation**.

| Model | Mean CV Macro F1 | Standard Deviation |
|---|---:|---:|
| Logistic Regression | **0.437** | 0.087 |
| Random Forest | 0.331 | 0.038 |
| CatBoost | 0.305 | 0.096 |

Cross-validation scores were lower than the single test-set scores, demonstrating that model performance was sensitive to how the small dataset was divided.

Logistic Regression remained the strongest model across multiple validation folds.

---

## 8. Hyperparameter Tuning

GridSearchCV with Stratified 5-Fold Cross-Validation and Macro F1 scoring was used to tune the shortlisted models.

### Tuned Logistic Regression

Best parameter:

- `C = 0.01`

Best cross-validation Macro F1:

- **0.529**

The low value of `C` indicates that stronger regularization produced better generalization.

### Tuned Random Forest

Best parameters:

- `n_estimators = 200`
- `max_depth = 5`
- `min_samples_split = 5`
- `min_samples_leaf = 2`

Best cross-validation Macro F1:

- **0.405**

Restricting tree complexity reduced overfitting and improved cross-validation performance.

### Tuned CatBoost

Best parameters:

- `iterations = 100`
- `depth = 3`
- `learning_rate = 0.05`

Best cross-validation Macro F1:

- **0.433**

The shallow depth and moderate learning rate improved generalization compared with the baseline CatBoost model.

---

## 9. Final Model Comparison

| Model | Baseline Test F1 | Baseline CV F1 | Tuned CV F1 |
|---|---:|---:|---:|
| Logistic Regression | **0.506** | **0.437** | **0.529** |
| CatBoost | 0.484 | 0.305 | 0.433 |
| Random Forest | 0.489 | 0.331 | 0.405 |

Tuned Logistic Regression achieved the highest cross-validation Macro F1 score of **0.529** and was selected as the final model.

Its performance suggests that the available relationships were captured more effectively by a regularized linear model than by more complex tree-based algorithms.

---

## 10. Final Conclusion

The project successfully developed and compared several multiclass classification models for Teaching Assistant evaluation prediction.

Tuned Logistic Regression was selected as the final model because it achieved the strongest and most consistent Macro F1 performance during hyperparameter tuning and cross-validation.

The final result also demonstrates that a more complex algorithm does not automatically produce better performance. Random Forest, XGBoost, and CatBoost were unable to outperform the regularized Logistic Regression model on this dataset.

The strongest predictive evidence was associated with English-speaking status and semester type, while class size showed limited predictive influence.

---

## 11. Dataset Limitations

The main limitation of this project was the very small dataset.

After duplicate removal, only **110 observations** remained. This created several challenges:

- Model scores were sensitive to different train-test divisions.
- Some instructors and courses had very few observations.
- Complex models had a higher risk of overfitting.
- Hyperparameter search spaces had to remain controlled.
- The resulting model cannot be considered production-ready.
- The findings may not generalize to other universities, courses, or Teaching Assistant populations.

Therefore, the reported performance should be interpreted as an exploratory result rather than evidence of reliable real-world deployment performance.

---

## 12. Future Improvements

Future work should focus primarily on collecting more high-quality observations rather than adding increasingly complex algorithms.

Recommended improvements include:

1. Collecting data across additional semesters, courses, instructors, and institutions.
2. Including Teaching Assistant experience, academic qualifications, student attendance, course difficulty, and assessment type.
3. Testing one-hot encoding for nominal identifiers such as course and instructor.
4. Evaluating ordinal classification methods if the three evaluation levels have a meaningful order.
5. Using nested cross-validation when a larger dataset becomes available.
6. Examining probability calibration before using model probabilities in real decisions.
7. Conducting fairness analysis to ensure that language-related variables do not create biased decisions.

---

## 13. Final Recommendation

Tuned Logistic Regression should be retained as the final model for this project because it achieved the highest cross-validation Macro F1 score while remaining simple, interpretable, and computationally efficient.

However, the model should not be used as an automated system for making decisions about Teaching Assistants. It is more appropriate as an academic demonstration of multiclass classification and model comparison.

The most valuable next step would be to collect a substantially larger and more representative dataset before considering deployment or further model optimization.